# Task 1A — Exploratory Data Analysis

The dataset is in **long format**: each row is a single (user, timestamp, variable, value) observation.  
Before any modelling, we must understand the temporal structure — because predicting next-day mood depends entirely on it.

Key questions:
1. How many users / variables / records?
2. How many days per user, and are they balanced?
3. How many readings per variable per user-day?
4. Are there temporal gaps (missing days) in each user's series?
5. What are the value ranges, and are there obvious outliers?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.utils import save_figure

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

## 1. Load raw data

In [ ]:
df = pd.read_csv('../data/raw/dataset_mood_smartphone.csv')
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date
df = df.drop(columns=['Unnamed: 0'])   # index column, not needed

print(f'Total rows : {len(df):,}')
print(f'Columns    : {df.columns.tolist()}')
print(f'Null values: {df.isnull().sum().to_dict()}')
df.head(10)

## 2. Dataset overview — summary statistics table

**Key structural facts**
- 27 users, 19 variables, 376,912 rows
- Only 202 null values (all in `value` column) — but "missing" data is implicit: a variable simply has no row for a given day
- `call` and `sms` are always 1.0 — they are event indicators, not durations

In [ ]:
print(f"Users     : {df['id'].nunique()}")
print(f"Variables : {df['variable'].nunique()}")
print()
print(df['variable'].unique())

In [ ]:
# Records per variable
rec_per_var = df.groupby('variable').size().sort_values(ascending=False).rename('n_records')

# Value statistics per variable
stats = df.groupby('variable')['value'].agg(['min', 'max', 'mean', 'std']).round(3)
stats.columns = ['min', 'max', 'mean', 'std']

summary = pd.concat([rec_per_var, stats], axis=1)
summary

**Notable observations from the table:**
- `screen` and `appCat.builtin` dominate the record count (sensor data sampled more frequently)
- `appCat.builtin` has a **minimum of −82,798** — clearly erroneous (will be addressed in Task 1B)
- `appCat.entertainment` also has a small negative value
- Several appCat variables have very large max values (tens of thousands of seconds)
- `mood` ranges 1–10 as expected; `circumplex.*` ranges −2 to 2 as expected

## 3. Temporal structure — days per user

In [ ]:
ranges = df.groupby('id')['date'].agg(['min', 'max'])
ranges['span_days']   = (pd.to_datetime(ranges['max']) - pd.to_datetime(ranges['min'])).dt.days + 1
ranges['active_days'] = df.groupby('id')['date'].nunique()
ranges['missing_days'] = ranges['span_days'] - ranges['active_days']
ranges['missing_pct']  = (ranges['missing_days'] / ranges['span_days'] * 100).round(1)
ranges.index.name = 'user'
ranges

In [ ]:
print('Active days per user:')
print(ranges['active_days'].describe().round(1))
print(f"\nMissing day % — min: {ranges['missing_pct'].min()}%  max: {ranges['missing_pct'].max()}%  mean: {ranges['missing_pct'].mean():.1f}%")

**Key finding:** Users are *not* balanced. Active days range from **50 to 101**. Up to **28.4% of days are missing** (AS14.28), which must be handled carefully in Task 1B before building the sliding window.

## 4. Mood readings per user-day (is mood measured once or many times?)

In [ ]:
mood_df = df[df['variable'] == 'mood']
readings_per_day = mood_df.groupby(['id', 'date']).size()

print('Mood readings per user-day:')
print(readings_per_day.describe().round(2))
print()
print('Distribution of reading counts:')
print(readings_per_day.value_counts().sort_index())

**Key finding:** Mood is recorded **multiple times per day** (1–6 times, most often 4–5 times).  
The target (next-day mood) must therefore be defined as the **daily average** of all mood readings — this is also what the assignment specifies.

## 5. Plots

In [ ]:
# --- Plot 1: Distribution of mood scores ---
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(mood_df['value'], bins=18, kde=True, ax=ax, color='steelblue')
ax.set_title('Distribution of Mood Scores (all users, all days)')
ax.set_xlabel('Mood (1–10)')
ax.set_ylabel('Count')
save_figure('1a_mood_distribution.png')
plt.show()

In [ ]:
# --- Plot 2: Active days and missing days per user ---
fig, ax = plt.subplots(figsize=(12, 4))
users = ranges.index
ax.bar(users, ranges['active_days'], label='Active days', color='steelblue')
ax.bar(users, ranges['missing_days'], bottom=ranges['active_days'], label='Missing days', color='salmon')
ax.set_xlabel('User')
ax.set_ylabel('Days')
ax.set_title('Active vs Missing Days per User')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_figure('1a_active_missing_days.png')
plt.show()

In [ ]:
# --- Plot 3: Records per variable (log scale to handle imbalance) ---
fig, ax = plt.subplots(figsize=(10, 5))
rec_per_var.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of records (log scale)')
ax.set_title('Record count per variable')
ax.set_xscale('log')
plt.tight_layout()
save_figure('1a_records_per_variable.png')
plt.show()

In [ ]:
# --- Plot 4: Mean daily mood over time for a sample of users ---
daily_mood = mood_df.groupby(['id', 'date'])['value'].mean().reset_index()
daily_mood['date'] = pd.to_datetime(daily_mood['date'])

sample_users = ['AS14.01', 'AS14.07', 'AS14.26', 'AS14.28']
fig, axes = plt.subplots(len(sample_users), 1, figsize=(12, 8), sharex=False)

for ax, user in zip(axes, sample_users):
    user_data = daily_mood[daily_mood['id'] == user]
    ax.plot(user_data['date'], user_data['value'], marker='o', markersize=3, linewidth=1)
    ax.set_ylabel('Mood')
    ax.set_ylim(1, 10)
    ax.set_title(f'User {user}')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.suptitle('Daily Average Mood Over Time (sample users)', fontsize=13)
plt.tight_layout()
save_figure('1a_mood_timeseries.png')
plt.show()

In [ ]:
# --- Plot 5: Boxplots of value distributions per variable (excluding call/sms) ---
exclude = ['call', 'sms']
plot_df = df[~df['variable'].isin(exclude)]

fig, ax = plt.subplots(figsize=(14, 6))
plot_df.boxplot(column='value', by='variable', ax=ax, rot=45, flierprops=dict(marker='.', markersize=2))
ax.set_title('Value distributions per variable')
ax.set_xlabel('Variable')
ax.set_ylabel('Value')
plt.suptitle('')
plt.tight_layout()
save_figure('1a_value_distributions.png')
plt.show()